In [1]:
import pandas as pd
import keras 
import numpy as np 

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Bidirectional
from tensorflow.keras.models import Model

import nltk
import string
from nltk.corpus import stopwords,re
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
train = pd.read_parquet(r"C:\Users\Krish\Downloads\Deep Learning\Attention_and_encoder_decoder\data\train-00000-of-00001.parquet")
test = pd.read_parquet(r"C:\Users\Krish\Downloads\Deep Learning\Attention_and_encoder_decoder\data\test-00000-of-00001.parquet")
validate = pd.read_parquet(r"C:\Users\Krish\Downloads\Deep Learning\Attention_and_encoder_decoder\data\validation-00000-of-00001.parquet")

In [3]:
train_small = train.sample(
    n=100000,
    random_state=42
)

In [4]:
print(train_small[1:2])
print(test[1:2])
print(validate[1:2])

                                              translation
643856  {'en': 'IDPL Chairman Major - General - LRB - ...
                                         translation
1  {'en': 'As America's road planners struggle to...
                                         translation
1  {'en': 'With encouragement from Principal Sand...


In [5]:
train_small["english"] = train_small["translation"].apply(lambda x: x["en"])
train_small["hindi"] = train_small["translation"].apply(lambda x: x["hi"])

## Preprocessing methods

In [6]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

def rem_stopwords(text):
    words = stopwords.words('english')

    return " ".join(
        word for word in text.split() if word.lower() not in words
    )

def to_lower(text):
    return "".join(
        text.lower()
    )

def rem_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

def rem_punctuations(text):
    exclude = string.punctuation
    return text.translate(str.maketrans('', '', exclude))


lemmatizer = WordNetLemmatizer()

def lemmatization(text):
    text = text.lower()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t.isalpha()]
    return " ".join(tokens)

def tokenize(text):
    tokenizer = word_tokenize(text)
    return tokenizer

import re

def clean_english(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def clean_hindi(text):
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Krish\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Krish\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Krish\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
train_small["english"] = train_small["english"].apply(rem_stopwords)
train_small["english"] = train_small["english"].apply(rem_tags)
train_small["english"] = train_small["english"].apply(rem_punctuations)

In [8]:
train_small["english"] = train_small["english"].apply(to_lower)

In [9]:
train_small["english"] = train_small["english"].apply(lemmatization)

In [10]:
train_small["english"] = train_small["english"].apply(
    clean_english
)

train_small["hindi"] = train_small["hindi"].apply(
    clean_hindi
)

In [11]:
train_small["hindi"] = train_small["hindi"].apply(
    lambda x: "<start> " + x + " <end>"
)

## Building tokenizer

In [ ]:


eng_tokenizer = Tokenizer(
    filters=""
)

hin_tokenizer = Tokenizer(
    filters=""
)

eng_tokenizer.fit_on_texts(
    train_small["english"]
)

hin_tokenizer.fit_on_texts(
    train_small["hindi"]
)

In [41]:
encoder_sequences = eng_tokenizer.texts_to_sequences(
    train_small["english"]
)

decoder_sequences = hin_tokenizer.texts_to_sequences(
    train_small["hindi"]
)

In [42]:
eng_vocab_size = len(eng_tokenizer.word_index) + 1
hin_vocab_size = len(hin_tokenizer.word_index) + 1

print("English vocabulary:", eng_vocab_size)
print("Hindi vocabulary:", hin_vocab_size)

English vocabulary: 50107
Hindi vocabulary: 109762


## Padding

In [15]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_encoder_len = max(
    len(x) for x in encoder_sequences
)

max_decoder_len = max(
    len(x) for x in decoder_sequences
)

encoder_input_data = pad_sequences(
    encoder_sequences,
    maxlen=max_encoder_len,
    padding="post"
)

decoder_sequences = pad_sequences(
    decoder_sequences,
    maxlen=max_decoder_len,
    padding="post"
)

In [16]:
decoder_input_data = decoder_sequences[:, :-1]

decoder_target_data = decoder_sequences[:, 1:]

In [17]:
decoder_target_data = decoder_target_data[..., np.newaxis]

In [18]:
embedding_dim = 256
latent_dim = 256

In [19]:
encoder_inputs = Input(
    shape=(None,),
    name="encoder_inputs"
)

In [20]:
encoder_embedding = Embedding(
    input_dim=eng_vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="encoder_embedding"
)(encoder_inputs)

In [43]:
encoder_lstm = Bidirectional(
    LSTM(
        latent_dim,
        return_sequences=True,
        return_state=True
    ),
    name="encoder_lstm"
)

encoder_outputs, forward_h, forward_c, backward_h, backward_c = encoder_lstm(
    encoder_embedding
)

In [44]:
encoder_embedding = Embedding(
    input_dim=eng_vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="encoder_embedding"
)(encoder_inputs)

## Decoder

In [45]:
decoder_inputs = Input(
    shape=(None,),
    name="decoder_inputs"
)

In [46]:
decoder_embedding_layer = Embedding(
    input_dim=hin_vocab_size,
    output_dim=embedding_dim,
    mask_zero=True,
    name="decoder_embedding"
)

decoder_embedding = decoder_embedding_layer(
    decoder_inputs
)

In [48]:
decoder_target = np.zeros((decoder_input_data.shape[0], decoder_input_data.shape[1], 1))
decoder_target[:, 0:-1, 0] = decoder_input_data[:, 1:]

In [49]:
encoder_inputs = Input(shape=(None,))
enc_emb = Embedding(eng_vocab_size, latent_dim)(encoder_inputs)
enc_outputs, state_h, state_c = LSTM(latent_dim, return_state=True)(enc_emb)
encoder_states = [state_h, state_c]

In [50]:
decoder_inputs = Input(shape=(None,))
dec_emb_layer = Embedding(hin_vocab_size, latent_dim)
dec_emb = dec_emb_layer(decoder_inputs)
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(hin_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [51]:
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='sparse_categorical_crossentropy')
model.fit([encoder_input_data, decoder_input_data], decoder_target, batch_size=64, epochs=20, validation_split=0.2)

Epoch 1/20


: 